# 03 – Histograms and Distributions

When you have a column of numbers — student marks, salaries, temperatures, delivery times — the first visual question is:

*What does the shape of this data look like?*

Is it spread out evenly? Is it bunched up in the middle? Is it skewed to one side? Does it have two humps?

A **histogram** answers all of these questions by showing how the data is distributed across a range of values.

This notebook covers:
- Histograms — the workhorse of distribution analysis
- Density plots (KDE) — smooth version of histograms
- Boxplots — outliers at a glance
- Common distribution shapes and what they mean

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Rebuild the student dataset from notebook 01
rng = np.random.default_rng(42)
n = 200

df = pd.DataFrame({
    "Student_ID": range(1001, 1001 + n),
    "Gender":     rng.choice(["Male","Female"], size=n, p=[0.52, 0.48]),
    "Department": rng.choice(["Science","Arts","Commerce"], size=n, p=[0.4,0.3,0.3]),
    "City":       rng.choice(["Mumbai","Pune","Delhi","Bangalore","Chennai"], size=n),
    "Study_Hours": np.clip(rng.normal(5, 2, n).round(1), 0, 12),
    "Attendance":  np.clip(rng.normal(75, 15, n).round(1), 20, 100),
    "Math":        np.clip(rng.normal(68, 18, n).round(0).astype(int), 0, 100),
    "Science":     np.clip(rng.normal(65, 20, n).round(0).astype(int), 0, 100),
    "English":     np.clip(rng.normal(70, 15, n).round(0).astype(int), 0, 100),
    "Fees_Paid":   rng.choice([True, False], size=n, p=[0.78, 0.22]),
})
for col, frac in [("Study_Hours",0.05),("Attendance",0.03),("Math",0.04),("Science",0.04),("English",0.03)]:
    idx = rng.choice(n, size=int(n * frac), replace=False)
    df.loc[idx, col] = np.nan
df.loc[rng.choice(n, 3, replace=False), "Math"] = rng.integers(0, 15, 3)
df["Total"] = df[["Math","Science","English"]].sum(axis=1)

print("Dataset ready. Shape:", df.shape)

We're using both Matplotlib and **Seaborn** from now on. Seaborn is a higher-level library built on top of Matplotlib — its default charts look better with less code, and it has specific functions for statistical plots like distributions, boxplots, and heatmaps.

## 1) The Basic Histogram

A histogram divides your data into **bins** (equal-width intervals) and counts how many values fall into each bin.

The x-axis is the value range, the y-axis is the count (or frequency).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(df["Math"].dropna(), bins=20, color="steelblue",
        edgecolor="white", linewidth=0.5)

ax.set_title("Distribution of Math Scores", fontsize=14, fontweight="bold")
ax.set_xlabel("Math Score")
ax.set_ylabel("Number of Students")
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/hist_math.png", dpi=120)
plt.close()
print("Histogram saved.")

**Reading a histogram:**
- The tallest bar shows where most values cluster
- Bars on the far left or right show rare extreme values
- A symmetric, bell-shaped curve means the data is roughly normally distributed
- An asymmetric shape means it's skewed

**Choosing bins:** `bins=20` means 20 intervals. Too few bins (like 5) and you lose detail. Too many (like 100) and you see noise instead of pattern. For 200 data points, 15-25 bins usually works well.

In [ ]:
# Add mean and median lines to understand skew
math_clean = df["Math"].dropna()
mean_val   = math_clean.mean()
median_val = math_clean.median()

fig, ax = plt.subplots(figsize=(8, 5))

ax.hist(math_clean, bins=20, color="steelblue", edgecolor="white", linewidth=0.5, alpha=0.8)

# Vertical reference lines
ax.axvline(mean_val,   color="firebrick", linestyle="--", linewidth=2,
           label=f"Mean: {mean_val:.1f}")
ax.axvline(median_val, color="darkorange", linestyle="-",  linewidth=2,
           label=f"Median: {median_val:.1f}")

ax.set_title("Math Score Distribution with Mean and Median", fontsize=13, fontweight="bold")
ax.set_xlabel("Math Score")
ax.set_ylabel("Number of Students")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/hist_math_lines.png", dpi=120)
plt.close()
print("Histogram with reference lines saved.")

**Mean vs Median on a histogram:**

If the mean and median are in roughly the same place, the distribution is roughly symmetric.

If the mean is **to the right** of the median, the distribution is **right-skewed** — a few very high values are pulling the mean up.

If the mean is **to the left** of the median, the distribution is **left-skewed** — a few very low values are pulling the mean down.

`axvline()` draws a vertical line at a given x position — very useful for marking thresholds, means, and cutoffs.

## 2) Distribution Shapes — What They Mean

Understanding the shape of a distribution is one of the most valuable skills in EDA. Here's a quick visual guide:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

rng_local = np.random.default_rng(1)

# Normal distribution
data_normal = rng_local.normal(50, 10, 500)
axes[0].hist(data_normal, bins=25, color="steelblue", edgecolor="white")
axes[0].set_title("Normal\n(Symmetric)", fontweight="bold")
axes[0].set_xlabel("Value")

# Right-skewed (positive skew)
data_right = rng_local.exponential(scale=20, size=500)
axes[1].hist(data_right, bins=25, color="darkorange", edgecolor="white")
axes[1].set_title("Right-Skewed\n(Long tail on right)", fontweight="bold")
axes[1].set_xlabel("Value")

# Left-skewed (negative skew)
data_left = 100 - rng_local.exponential(scale=20, size=500)
axes[2].hist(data_left, bins=25, color="seagreen", edgecolor="white")
axes[2].set_title("Left-Skewed\n(Long tail on left)", fontweight="bold")
axes[2].set_xlabel("Value")

# Bimodal (two peaks)
data_bimodal = np.concatenate([rng_local.normal(30, 5, 250), rng_local.normal(70, 5, 250)])
axes[3].hist(data_bimodal, bins=25, color="purple", edgecolor="white")
axes[3].set_title("Bimodal\n(Two peaks)", fontweight="bold")
axes[3].set_xlabel("Value")

for ax in axes:
    ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle("Common Distribution Shapes", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/hist_shapes.png", dpi=120, bbox_inches="tight")
plt.close()
print("Distribution shapes chart saved.")

**What each shape means in practice:**

- **Normal (symmetric):** Rare in pure form, but many real variables approximate it. Height, exam scores in large populations, measurement errors.
- **Right-skewed:** Very common. Income distributions (most people earn moderate amounts, a few earn enormous amounts). Prices, delays, number of purchases.
- **Left-skewed:** Less common. Age at retirement, scores on easy exams (most students score high, few score very low).
- **Bimodal (two peaks):** Often means your data has two distinct groups that you should separate. For example, a bimodal height distribution might be mixing males and females.

## 3) Comparing Multiple Distributions — Overlapping Histograms

When you want to compare distributions across groups, put them in the same chart with transparency.

In [ ]:
# Math score distribution by department
fig, ax = plt.subplots(figsize=(9, 5))

for dept, colour in [("Science","steelblue"), ("Arts","darkorange"), ("Commerce","seagreen")]:
    data = df[df["Department"] == dept]["Math"].dropna()
    ax.hist(data, bins=15, alpha=0.5, color=colour, edgecolor="white", label=dept)

ax.set_title("Math Score Distribution by Department", fontsize=13, fontweight="bold")
ax.set_xlabel("Math Score")
ax.set_ylabel("Number of Students")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/hist_overlapping.png", dpi=120)
plt.close()
print("Overlapping histograms saved.")

`alpha=0.5` (50% transparency) lets you see where histograms overlap. Where two colours mix, both distributions have values there.

If one group's histogram sits clearly to the right of another's, that group tends to score higher. If they overlap heavily, the distributions are similar.

## 4) KDE Plot — Smooth Distribution Curve

A **Kernel Density Estimate (KDE)** is a smooth curve version of a histogram. Instead of bars, you get a continuous line that shows the probability density of the data.

Seaborn makes this very easy with `sns.kdeplot()`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

for dept, colour in [("Science","steelblue"), ("Arts","darkorange"), ("Commerce","seagreen")]:
    data = df[df["Department"] == dept]["Math"].dropna()
    sns.kdeplot(data, ax=ax, color=colour, linewidth=2.5, label=dept, fill=True, alpha=0.1)

ax.set_title("Math Score Distribution (KDE) by Department", fontsize=13, fontweight="bold")
ax.set_xlabel("Math Score")
ax.set_ylabel("Density")
ax.legend()
ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/kde_dept.png", dpi=120)
plt.close()
print("KDE plot saved.")

KDE plots are better for comparing distributions because the smooth curves don't depend on bin size. Three departments overlap heavily in the middle — confirming that department alone doesn't strongly predict Math performance.

`fill=True` shades the area under the curve, making group differences easier to see.

## 5) Histogram + KDE Together (Seaborn `histplot`)

Seaborn's `histplot()` can overlay both a histogram and a KDE in one call — the most efficient way to understand a distribution.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, col, colour in zip(axes, ["Math","Science","English"], ["steelblue","darkorange","seagreen"]):
    sns.histplot(df[col].dropna(), bins=20, kde=True, color=colour, 
                 edgecolor="white", ax=ax, alpha=0.7)
    
    mean_v = df[col].mean()
    ax.axvline(mean_v, color="black", linestyle="--", linewidth=1.5, label=f"Mean: {mean_v:.1f}")
    ax.set_title(f"{col} Score Distribution", fontweight="bold")
    ax.set_xlabel("Score")
    ax.legend(fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle("Score Distributions Across All Subjects", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/histplot_all.png", dpi=120, bbox_inches="tight")
plt.close()
print("Combined histplot saved.")

`kde=True` adds the smooth density curve on top of the histogram bars automatically. This is the cleanest way to show a full distribution in a single chart.

## 6) Boxplot — Outliers at a Glance

A **boxplot** (also called a box-and-whisker plot) summarises a distribution in 5 numbers:

```
     ┌──────────────────────────────┐
     │         │       │            │
 ─●─ │   Q1    │Median │    Q3      │ ─────── whisker ── ●  outlier
     │         │       │            │
     └──────────────────────────────┘

● = outlier (beyond 1.5 × IQR from box edges)
```

- **Box:** shows the middle 50% of data (Q1 to Q3)
- **Line in box:** the median (Q2)
- **Whiskers:** extend to min/max within 1.5 × IQR
- **Dots beyond whiskers:** outliers

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

data_to_plot = [df["Math"].dropna(), df["Science"].dropna(), df["English"].dropna()]
labels = ["Math", "Science", "English"]

bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True,
                medianprops={"color":"black","linewidth":2})

# Colour each box
colours = ["steelblue", "darkorange", "seagreen"]
for patch, colour in zip(bp["boxes"], colours):
    patch.set_facecolor(colour)
    patch.set_alpha(0.7)

ax.set_title("Score Distribution by Subject", fontsize=14, fontweight="bold")
ax.set_ylabel("Score")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/boxplot_subjects.png", dpi=120)
plt.close()
print("Boxplot saved.")

Boxplots let you compare distributions side by side at a glance:
- Which subject has the highest median?
- Which subject has the widest spread (tall box = high variability)?
- Which subject has the most outliers (dots beyond the whiskers)?

The outlier dots we injected into `Math` will be visible as points far below the lower whisker.

In [ ]:
# Grouped boxplot with Seaborn — distribution by department
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(data=df, x="Department", y="Math", hue="Department",
            palette={"Science":"steelblue","Arts":"darkorange","Commerce":"seagreen"},
            legend=False, ax=ax)

ax.set_title("Math Score Distribution by Department", fontsize=13, fontweight="bold")
ax.set_xlabel("Department")
ax.set_ylabel("Math Score")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/boxplot_dept.png", dpi=120)
plt.close()
print("Grouped boxplot saved.")

Seaborn's `boxplot()` automatically groups by a categorical column when you pass `x="Department"`. This is much faster than creating separate boxplots manually.

`palette` sets the colour for each category value.

## 7) Violin Plot — Box + Distribution Together

A **violin plot** combines a boxplot's summary statistics with the KDE's shape information. The width of the violin at any point shows how many data points are there.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.violinplot(data=df, x="Department", y="Math", hue="Department",
               palette={"Science":"steelblue","Arts":"darkorange","Commerce":"seagreen"},
               inner="box",    # show a mini boxplot inside the violin
               legend=False, ax=ax)

ax.set_title("Math Score Distribution by Department (Violin)", fontsize=13, fontweight="bold")
ax.set_xlabel("Department")
ax.set_ylabel("Math Score")
ax.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("/home/claude/05_eda/violin_dept.png", dpi=120)
plt.close()
print("Violin plot saved.")

Violin plots give more information than boxplots — you can see the full shape of the distribution, not just the summary statistics. They're especially useful for detecting bimodal distributions (two humps) that a boxplot would completely hide.

`inner="box"` draws a miniature boxplot inside the violin, so you get both the shape and the summary.

## 8) Putting It Together — Full Distribution Analysis

Let's do a complete distribution analysis of the Attendance column — a real analysis workflow.

In [ ]:
attendance = df["Attendance"].dropna()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Histogram + KDE
sns.histplot(attendance, bins=20, kde=True, color="steelblue",
             edgecolor="white", alpha=0.7, ax=axes[0])
axes[0].axvline(attendance.mean(),   color="firebrick", linestyle="--", linewidth=2,
                label=f"Mean: {attendance.mean():.1f}%")
axes[0].axvline(attendance.median(), color="darkorange", linestyle="-",  linewidth=2,
                label=f"Median: {attendance.median():.1f}%")
axes[0].axvline(75, color="black", linestyle=":", linewidth=1.5, label="75% threshold")
axes[0].set_title("Attendance Distribution", fontweight="bold")
axes[0].set_xlabel("Attendance %")
axes[0].legend(fontsize=8)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)

# 2. Boxplot by Department
sns.boxplot(data=df, x="Department", y="Attendance", hue="Department",
            palette={"Science":"steelblue","Arts":"darkorange","Commerce":"seagreen"},
            legend=False, ax=axes[1])
axes[1].set_title("Attendance by Department", fontweight="bold")
axes[1].set_xlabel("Department")
axes[1].set_ylabel("Attendance %")
axes[1].grid(axis="y", linestyle="--", alpha=0.4)

# 3. Boxplot by Gender
sns.boxplot(data=df, x="Gender", y="Attendance", hue="Gender",
            palette={"Male":"steelblue","Female":"darkorange"},
            legend=False, ax=axes[2])
axes[2].set_title("Attendance by Gender", fontweight="bold")
axes[2].set_xlabel("Gender")
axes[2].set_ylabel("Attendance %")
axes[2].grid(axis="y", linestyle="--", alpha=0.4)

fig.suptitle("Attendance — Full Distribution Analysis", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("/home/claude/05_eda/attendance_analysis.png", dpi=120, bbox_inches="tight")
plt.close()
print("Full distribution analysis saved.")

Three panels, three questions answered:
1. What does the attendance distribution look like overall? (left)
2. Does attendance differ by department? (middle)
3. Does attendance differ by gender? (right)

The dotted vertical line at 75% marks the threshold for exam eligibility in most Indian colleges — instantly visible how many students fall below it.

## Summary

| Chart | Best for |
|---|---|
| `ax.hist()` | Quick histogram |
| `sns.histplot(kde=True)` | Histogram + smooth curve in one shot |
| `sns.kdeplot()` | Comparing multiple distributions as curves |
| `ax.boxplot()` | Outliers + quartiles for one or more groups |
| `sns.boxplot()` | Grouped boxplot by category (much easier) |
| `sns.violinplot()` | Full shape + summary statistics together |

**Key insights to look for:**
- Where is the bulk of the data? (the tall bars / wide part of violin)
- Is it symmetric or skewed?
- Are there outliers? (dots beyond boxplot whiskers)
- Does the distribution differ across groups?

Next up: **Scatter Plots and Correlation** — exploring relationships between pairs of columns.

## Practice Questions 1

1. Plot the distribution of `Study_Hours` using `sns.histplot(kde=True)`. Add a vertical line at the mean.
2. Is the distribution symmetric? Where is the bulk of the data?
3. Create a boxplot comparing `Study_Hours` across departments. Do departments differ much?

## Practice Questions 2

1. Create a side-by-side violin plot comparing the `Total` score distribution for students who paid fees vs those who didn't (use `Fees_Paid` as the x-axis).
2. Add a horizontal line at `Total = 150` (which represents 50% marks across all three subjects).
3. What does the shape of the violin tell you that a boxplot wouldn't?